In [33]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import os
import shutil
import random

import torch
import torch.nn as nn
import torch.nn.init as init
from torchvision import transforms, datasets, models

In [6]:
df = pd.read_csv("data/brand_info.csv", index_col=0) # We don't need the unnamed column
df.head()

,ID,GenderType,Type,SubType,Article,PrimaryColor,Seasonal,Year,Use,Brand
1,39386,Men,Apparel,Bottomwear,Jeans,Blue,Summer,2012.0,Casual,Peter England
2,59263,Women,Accessories,Watches,Watches,Silver,Winter,2016.0,Casual,Titan
4,53759,Men,Apparel,Topwear,Tshirts,Grey,Summer,2012.0,Casual,Puma
8,29114,Men,Accessories,Socks,Socks,Navy Blue,Summer,2012.0,Casual,Puma
10,9204,Men,Footwear,Shoes,Casual Shoes,Black,Summer,2011.0,Casual,Puma


In [7]:
df.shape

(15137, 10)

# **MODEL 1:** `Data Preprocessing` 

## Dataset Examinations:

> We have a folder with 44,327 RGB images, each image of 60x80(Wxh). The ID column represents the name of the image file. This means we do not have to annotate the labels for each image and can simply use these rows as the annotations for each image. 

> There are only 15137 rows in the csv file which means we only have labels for these files. Rest of the image files are unlabelled. This explains that only 34.14% of the images are labeled. According to the project guidelines, we will consider rest images as "fake" and ignore them since these images have been uploaded with incorrect brand info by the 3rd party sellers.

> `This observation implies that the 34.14% images data are in the csv file which can be now tagged as "Genuine" and rest of the images data can be tagged as "Fake". This will make our class unbalanced but it is (65% to 35%) which is not huge difference and we can say this will not affect or introduce bias in the results.`

> We will build a CNN model to classify the real vs fake brands first. We will use opencv with pytorch framework for this task. When comparing two popular frameworks tensorflow and pytorch, the training time was significantly faster in pytorch while the memory usages were also high according to [this](https://arc.net/l/quote/ujakprto) article. The accuracy for both the frameworks were similar. Pytorch is more oriented towards python and is easy to interpret. So, we will use pytorch because of less training time and ease of use.

In [8]:
df.isnull().sum()

ID              0
GenderType      0
Type            0
SubType         0
Article         0
PrimaryColor    6
Seasonal        1
Year            1
Use             4
Brand           0
dtype: int64

> Since there are very less null values, we can drop them.

In [9]:
df.dropna(inplace=True)
(df.isna().sum(), df.shape)

(ID              0
 GenderType      0
 Type            0
 SubType         0
 Article         0
 PrimaryColor    0
 Seasonal        0
 Year            0
 Use             0
 Brand           0
 dtype: int64,
 (15126, 10))

In [10]:
df["ID"].duplicated().sum()

0

### Initial Observations:

- We have images in RGB, we can to convert it into grayscale, which will reduce the complexity. But the colors could be the features that determine the brand authenticity. For the purpose of this project, we will keep the images in RGB format because we might lose features if we convert them into grayscale.
- We need to create a labelled dataset for the image files which will contain the image files with their authenticity("Genuine", "Not Genuine"). We will load the files and then map the file names with the ID from the csv data. 
- We need to process the images to pass them through the CNN. We will convert them into 64x64 images now for simplicity. Based on the performance of model, we can increase this later.

### Since, pytorch provides option to resize the images, we will not resize using opencv right now.

### Create a labelled dataset for first CNN, annotating the images present in the "ID" column of csv file as "Genuine" and other image files as "Fake". 
### Now to do this, we can manually create a image-label pair and dump them in a csv file. However, pytorch happens to provide a powerful class to automatically label these images. We will be using the [ImageFolder](https://arc.net/l/quote/lpynlrki) class from pytorch to annotate the images. Steps to perform annotations will be:
- Identify the images names for "Genuine" class and put them in an array.
- Move all of the images having names that match the names in the create array in a folder "Genuine".
- Move rest of the images in the folder named "Fake".

In [11]:
genuine_ids_list = df["ID"].astype(str).to_list() # * According to the project guideline, all the IDs in csv files might not have corresponding images in the file.
len(genuine_ids_list)

15126

In [12]:
main_dir = "data/images"
genuine_dir = "data/new/genuine" # * Move to new folder
fake_dir = "data/new/fake"

os.makedirs(genuine_dir, exist_ok=True)
os.makedirs(fake_dir, exist_ok=True)


In [13]:
fake_ids = list() # * Will be used in train test split
genuine_ids = list() 
for filename in os.listdir(main_dir):
    splitted = filename.split(".")
    file_id = splitted[0]
    ext = splitted[1]

    # * Ensure the validity of images extension
    if ext not in ["jpg"]:
        print("Not valid image...")

    if file_id in genuine_ids_list:
        shutil.move(os.path.join(main_dir, filename), os.path.join(genuine_dir, filename))
        genuine_ids_list.remove(file_id) # * According to the project guideline, all the IDs in csv files might not have corresponding images in the file. Remaining items in the genuine_ids_list will be those IDs.
        genuine_ids.append(file_id)
    else:
        fake_ids.append(file_id)
        shutil.move(os.path.join(main_dir, filename), os.path.join(fake_dir, filename))

### These IDs below does not have corresponding Images in the images folder

In [14]:
genuine_ids_list

['1164',
 '1163',
 '31244',
 '31415',
 '5026',
 '31236',
 '31252',
 '31241',
 '31284',
 '31225',
 '12347']

In [15]:
genuine_size = len(os.listdir(genuine_dir))
fake_size = len(os.listdir(fake_dir))
print(f"Genuine images: {genuine_size}")
print(f"Fake images: {fake_size}")

Genuine images: 15115

Fake images: 29211


### So, the images are classified under Genuine and Fake, now we need to split them into train, test and validate dataset. We will create these subsets for each class. We will perform (80-18-2)% (train-test-validation) split. We will have 2% of totally unseen images by the model during the process of training and testing.

> We will have a train, test and validation folder inside data folder now

In [16]:

# * Creating directories for genuine class
genuine_train_dir = "data/train/genuine"
genuine_test_dir = "data/test/genuine"
genuine_val_dir = "data/val/genuine"

os.makedirs(genuine_train_dir, exist_ok=True)
os.makedirs(genuine_test_dir, exist_ok=True)
os.makedirs(genuine_val_dir, exist_ok=True)

# * Same for fake class
fake_train_dir = "data/train/fake"
fake_test_dir = "data/test/fake"
fake_val_dir = "data/val/fake"

os.makedirs(fake_train_dir, exist_ok=True)
os.makedirs(fake_test_dir, exist_ok=True)
os.makedirs(fake_val_dir, exist_ok=True)

In [17]:
print(len(genuine_ids), len(fake_ids))
genuine_size, fake_size

15115 29211


(15115, 29211)

> Genuine id list size is same as actual genuine folder file size. This means that each of our data has corresponding image file.

In [18]:
def get_ttv_size(total_size): # * (80-18-2) split
    return (int((80/100)*total_size), int((18/100)*total_size), int((2/100)*total_size))

# * Lets shuffle the list randomly to choose random images in train, test and val sets.
random.seed(42) # * This will be similar to stratified train_test_split with seed 42
random.shuffle(genuine_ids)
random.shuffle(fake_ids)

# * Dealing with genuine data split (genuine_dir)
genuine_train_size, genuine_test_size, genuine_val_size = get_ttv_size(genuine_size)

train_genuine_ids = genuine_ids[:genuine_train_size]
test_genuine_ids = genuine_ids[genuine_train_size:genuine_train_size+genuine_test_size]
val_genuine_ids = genuine_ids[genuine_train_size+genuine_test_size:]
    
print(f"Genuine Splits: {len(train_genuine_ids), len(test_genuine_ids), len(val_genuine_ids)}")

Genuine Splits: (12092, 2720, 303)


In [19]:
# * Dealing with fake data split (fake_dir)
fake_train_size, fake_test_size, fake_val_size = get_ttv_size(fake_size)

train_fake_ids = fake_ids[:fake_train_size]
test_fake_ids = fake_ids[fake_train_size:fake_train_size+fake_test_size]
val_fake_ids = fake_ids[fake_train_size+fake_test_size:]
    
print(f"Fake Splits: {len(train_fake_ids), len(test_fake_ids), len(val_fake_ids)}")

Fake Splits: (23368, 5257, 586)


In [20]:
root_directory = 'data'
current_directory = 'data/new' # * will be moving from current dir to root dir (train, test, val)
classes = ['genuine', 'fake']
splits = ['train', 'test', 'val']

ids_mapping = {
    'train': {'genuine': train_genuine_ids, 'fake': train_fake_ids},
    'test': {'genuine': test_genuine_ids, 'fake': test_fake_ids},
    'val': {'genuine': val_genuine_ids, 'fake': val_fake_ids},
}

for stage in splits: 
    for class_name in classes: 
        source_folder = os.path.join(current_directory, class_name)
        destination_folder = os.path.join(root_directory, stage, class_name)
        
        os.makedirs(destination_folder, exist_ok=True)
        
        for image_id in ids_mapping[stage][class_name]: # * Since it is dict inside dict, we have two levels
            filename = image_id + '.jpg' # * We have ensured that all of our image files are  of .jpg format
            source_file = os.path.join(source_folder, filename)
            destination_file = os.path.join(destination_folder, filename)
            
            shutil.move(source_file, destination_file)


print((len(os.listdir(genuine_train_dir))), (len(os.listdir(genuine_test_dir))), (len(os.listdir(genuine_val_dir))))
print((len(os.listdir(fake_train_dir))), (len(os.listdir(fake_test_dir))), (len(os.listdir(fake_val_dir))))

12092 2720 303

23368 5257 586


### Nice, these numbers(folder sizes for each folder) match our previous Genuine_splits and Fake Splits lists size. This means we have successfully moved respected percentage of files into train, test and val folders. XD

## Defining CNN, Training and Evaluations

In [ ]:

# * Setup torch device:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

#### We need transform operations to perform dataloading in pytorch. This is the step where we will augment the data, if we have to.

In [ ]:
data_transforms = transforms.Compose([
    transforms.Resize((64, 64)), # * We will use 64 by 64 as the images are in 80 by 60 format.
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(), # * We will convert the images to tensors.
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)) 
]
)

#### Now, we will setup the data loaders where we will load training and testing data and use data_transforms as argument for the data loader.

In [ ]:
train_loader = torch.utils.data.DataLoader(
    datasets.ImageFolder(root='/kaggle/input/true-brands/train', transform=data_transforms),
    batch_size=64, shuffle=True, num_workers=4)

test_loader = torch.utils.data.DataLoader(
    datasets.ImageFolder(root='/kaggle/input/true-brands/test', transform=data_transforms),
    batch_size=64, shuffle=True, num_workers=4)

# * Print the classes from DataLoader object
classes = train_loader.dataset.classes
print(classes)

### Define CNN model
> This architecture was changed many times during the project.

In [ ]:
class TBCNet(nn.Module): # * Our class will inherit from nn.Module class as base module.
    def __init__(self):
        super(TBCNet, self).__init__()

        # ! Convolution #1
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, stride=1, padding=1) # * We will use 32 out channels, higher the value, higher the features model has to learn and is computationally expensive. We will increase this value later if the problem seems to be more complex.
        self.bn1 = nn.BatchNorm2d(num_features=32) # * num_features will be the number of output channels.
        # * We need activation
        self.relu1 = nn.ReLU(inplace=True)

        # ! OP (bs, 32, 64, 64)

        # ! Pooling #1
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2) # * Reduce the size of image using 2 by 2 filter.
        # ! OP (bs, 32, 32, 32)

        # ! Convolution #2
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(num_features=64)
        self.relu2 = nn.ReLU(inplace=True)
        # ! OP (bs, 64, 32, 32)

        # ! Convolution #3
        self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, stride=1, padding=1)
        self.bn3 = nn.BatchNorm2d(num_features=128)
        self.relu3 = nn.ReLU(inplace=True)
        # ! OP (bs, 128, 32, 32)

        self.fc1 = nn.Linear(in_features=(128*32*32), out_features=2) # * The number if input features will be out_channels from previous layer * image size which will be 128*32*32 and the out_features will be 2 because we have 2 classes.

    # * Arrange the architecture to create Feed forward function
    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu1(x)

        x = self.pool1(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu2(x)

        x = self.conv3(x)
        x = self.bn3(x)
        x = self.relu3(x)

        # ! (bs, 128, 32, 32)
        x = x.view(-1, 128*32*32)
        x = self.fc1(x)

        return x

In [ ]:
tb_model = TBCNet().to(device)

#### Adam Optimizer and CE loss function

In [ ]:
# * optimizer and loss function #LR was changed many times
tb_optimizer = torch.optim.Adam(tb_model.parameters(), lr=0.01, weight_decay=0.0001) #! weight_decay is to regularize the weights, keep the weights small
tb_loss_function = nn.CrossEntropyLoss()

In [74]:
train_count= sum([len(everything) for _, _, everything in os.walk("/kaggle/input/true-brands/train")])
test_count= sum([len(everything) for _, _, everything in os.walk("/kaggle/input/true-brands/test")])
train_count, test_count

(35460, 7977)

## Training the ConvNet

In [66]:
num_epochs = 5
best_accuracy = 0.0

for epoch in range(num_epochs):
    tb_model.train()
    train_accuracy = 0.0
    train_loss = 0.0
    test_accuracy=0.0

    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device) #* Images and Labels are of specified batch_size in data loader.
        tb_optimizer.zero_grad() # * In each epoch we want to clear out the gradients calculated by optimizer. 
        # *Since, this is not RNN, we do not want the accumulated gradients in each epoch

        x=tb_model(images) #* Pass images through model to get predicted outputs x
        loss=tb_loss_function(x, labels) # * Now, we calculate the loss between output x and ground truth labels
        # * using cross entropy loss function defined earlier
        loss.backward() # * Go find the imposters, there are many
        tb_optimizer.step() #* Apply optimizer, update weights

        train_loss += loss.item() * images.size(0) #* This will be the total loss for the batch as we are increasing the value each time.
        _,prediction = torch.max(x, 1) # * Max value from the logits in x(op). Highest Probability of x values. Dimension 0 will be batch size and 1 will be logits

        train_accuracy += torch.sum(prediction == labels).item() #* Total sum of values where prediction are true
        # * .item() will convert tensor to scalar
        
    train_accuracy /= train_count #* Average accuracy and loss
    train_loss /= train_count
    
    tb_model.eval() # * Evaluation mode
    
    with torch.no_grad(): # * During testing, we can save on some memory by not calculating gradients
        for images, labels in test_loader: #* Sae stuff as training but fur testing data
            images, labels = images.to(device), labels.to(device)
            
            x = tb_model(images)
            _, prediction = torch.max(x, 1) #! 1 is the dimension for logits
            test_accuracy += torch.sum(prediction == labels).item()
    
    test_accuracy /= test_count
    
    print(f'Epoch: {epoch} Train Loss: {train_loss:.4f} Train Accuracy: {train_accuracy:.4f} Test Accuracy: {test_accuracy:.4f}')
    
    # * We can save the best weights
    if test_accuracy > best_accuracy:
        torch.save(tb_model.state_dict(), 'ckpt.torchmodel')
        best_accuracy = test_accuracy

Epoch: 0 Train Loss: 0.5345 Train Accuracy: 0.7283 Test Accuracy: 0.7218
Epoch: 1 Train Loss: 0.5331 Train Accuracy: 0.7276 Test Accuracy: 0.7278
Epoch: 2 Train Loss: 0.5290 Train Accuracy: 0.7312 Test Accuracy: 0.7233
Epoch: 3 Train Loss: 0.5301 Train Accuracy: 0.7292 Test Accuracy: 0.7282
Epoch: 4 Train Loss: 0.5288 Train Accuracy: 0.7309 Test Accuracy: 0.7265


In [21]:
num_epochs = 5

best_accuracy = 0.0

for epoch in range(num_epochs):
    tb_model.train()
    train_accuracy = 0.0
    test_accuracy=0.0
    train_loss = 0.0

    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        tb_optimizer.zero_grad()

        x=tb_model(images)
        loss=tb_loss_function(x, labels)
        loss.backward()
        tb_optimizer.step()

        train_loss += loss.item() * images.size(0)
        _,prediction = torch.max(x, 1)

        train_accuracy += torch.sum(prediction == labels).item()
        
    train_accuracy /= train_count
    train_loss /= train_count
    
    tb_model.eval()
    
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            
            x = tb_model(images)
            _, prediction = torch.max(x, 1)
            test_accuracy += torch.sum(prediction == labels).item()
    
    test_accuracy /= test_count
    
    print(f'Epoch: {epoch} Train Loss: {train_loss:.4f} Train Accuracy: {train_accuracy:.4f} Test Accuracy: {test_accuracy:.4f}')
    
    if test_accuracy > best_accuracy:
        torch.save(tb_model.state_dict(), 'ckpt.torchmodel')
        best_accuracy = test_accuracy

Epoch: 0 Train Loss: 5.4106 Train Accuracy: 0.6252 Test Accuracy: 0.6642
Epoch: 1 Train Loss: 0.6113 Train Accuracy: 0.6701 Test Accuracy: 0.6751
Epoch: 2 Train Loss: 0.5886 Train Accuracy: 0.6833 Test Accuracy: 0.6594
Epoch: 3 Train Loss: 0.7731 Train Accuracy: 0.6589 Test Accuracy: 0.6657
Epoch: 4 Train Loss: 0.6252 Train Accuracy: 0.6682 Test Accuracy: 0.6729


## Let's perform some more data augmentations, add More layers and perform He Initialization as well.

In [24]:

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

data_transforms = transforms.Compose([
    transforms.Resize((64, 64)), # * We will use 64 by 64 as the images are in 80 by 60 format.
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20), # * Adding stronger rotation augmentations
    transforms.RandomResizedCrop(64, scale=(0.8, 1.0)), # * Scaling & Cropping
    transforms.ToTensor(), # * We will convert the images to tensors.
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)) 
]
)

train_loader = torch.utils.data.DataLoader(
    datasets.ImageFolder(root='/kaggle/input/true-brands/train', transform=data_transforms),
    batch_size=64, shuffle=True, num_workers=6)

test_loader = torch.utils.data.DataLoader(
    datasets.ImageFolder(root='/kaggle/input/true-brands/test', transform=data_transforms),
    batch_size=64, shuffle=True, num_workers=6)

# * Print the classes from DataLoader object
classes = train_loader.dataset.classes
print(classes)

['fake', 'genuine']


In [29]:
class TBCNet(nn.Module): # * Our class will inherit from nn.Module class as base module.
    def __init__(self):
        super(TBCNet, self).__init__()

        # ! Convolution #1
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, stride=1, padding=1) # * We will use 32 out channels, higher the value, higher the features model has to learn and is computationally expensive. We will increase this value later if the problem seems to be more complex.
        init.kaiming_normal_(self.conv1.weight, mode='fan_out', nonlinearity='relu')
        self.bn1 = nn.BatchNorm2d(num_features=32) # * num_features will be the number of output channels.
        # * We need activation
        self.relu1 = nn.ReLU(inplace=True)

        # ! OP (bs, 32, 64, 64)

        # ! Pooling #1
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2) # * Reduce the size of image using 2 by 2 filter.
        # ! OP (bs, 32, 32, 32)

        # ! Convolution #2
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1)
        init.kaiming_normal_(self.conv2.weight, mode='fan_out', nonlinearity='relu')
        self.bn2 = nn.BatchNorm2d(num_features=64)
        self.relu2 = nn.ReLU(inplace=True)
        # ! OP (bs, 64, 32, 32)

        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2) 
        # ! OP (bs, 64, 16, 16)

        # ! Convolution #3
        self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, stride=1, padding=1)
        init.kaiming_normal_(self.conv3.weight, mode='fan_out', nonlinearity='relu')
        self.bn3 = nn.BatchNorm2d(num_features=128)
        self.relu3 = nn.ReLU(inplace=True)
        # ! OP (bs, 128, 16, 16)

        self.conv4 = nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, stride=1, padding=1)
        init.kaiming_normal_(self.conv4.weight, mode='fan_out', nonlinearity='relu')
        self.bn4 = nn.BatchNorm2d(num_features=256)
        self.relu4 = nn.ReLU(inplace=True)
        # ! OP (bs, 256, 16, 16)

        self.fc1 = nn.Linear(in_features=(256*16*16), out_features=2) # * The number if input features will be out_channels from previous layer * image size which will be 256*16*16 and the out_features will be 2 because we have 2 classes.

    # * Arrange the architecture to create Feed forward function
    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu1(x)

        x = self.pool1(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu2(x)

        x = self.pool2(x)

        x = self.conv3(x)
        x = self.bn3(x)
        x = self.relu3(x)

        x = self.conv4(x)
        x = self.bn4(x)
        x = self.relu4(x)

        # ! (bs, 256, 16, 16)
        x = x.view(-1, 256*16*16)
        x = self.fc1(x)

        return x
        
tb_model = TBCNet().to(device)

#optimizer and loss function
tb_optimizer = torch.optim.Adam(tb_model.parameters(), lr=0.01, weight_decay=0.0001)
tb_loss_function = nn.CrossEntropyLoss()

train_count= sum([len(everything) for _, _, everything in os.walk("/kaggle/input/true-brands/train")])
test_count= sum([len(everything) for _, _, everything in os.walk("/kaggle/input/true-brands/test")])
train_count, test_count

(35460, 7977)

In [30]:
num_epochs = 7

best_accuracy = 0.0

for epoch in range(num_epochs):
    tb_model.train()
    train_accuracy = 0.0
    train_loss = 0.0

    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        tb_optimizer.zero_grad()

        x=tb_model(images)
        loss=tb_loss_function(x, labels)
        loss.backward()
        tb_optimizer.step()

        train_loss += loss.item() * images.size(0)
        _,prediction = torch.max(x, 1)

        train_accuracy += torch.sum(prediction == labels).item()
        
    train_accuracy /= train_count
    train_loss /= train_count
    
    tb_model.eval()
    
    test_accuracy=0.0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            
            x = tb_model(images)
            _, prediction = torch.max(x, 1)
            test_accuracy += torch.sum(prediction == labels).item()
    
    test_accuracy /= test_count
    
    print(f'Epoch: {epoch} Train Loss: {train_loss:.4f} Train Accuracy: {train_accuracy:.4f} Test Accuracy: {test_accuracy:.4f}')

    tb_scheduler.step()
    
    if test_accuracy > best_accuracy:
        torch.save(tb_model.state_dict(), 'ckpt.torchmodel')
        best_accuracy = test_accuracy

Epoch: 0 Train Loss: 1.8183 Train Accuracy: 0.6372 Test Accuracy: 0.6271
Epoch: 1 Train Loss: 0.6047 Train Accuracy: 0.6699 Test Accuracy: 0.6589
Epoch: 2 Train Loss: 0.5909 Train Accuracy: 0.6817 Test Accuracy: 0.6927
Epoch: 3 Train Loss: 0.5737 Train Accuracy: 0.6922 Test Accuracy: 0.6939
Epoch: 4 Train Loss: 0.5645 Train Accuracy: 0.6937 Test Accuracy: 0.6984
Epoch: 5 Train Loss: 0.5616 Train Accuracy: 0.6961 Test Accuracy: 0.7019
Epoch: 6 Train Loss: 0.5617 Train Accuracy: 0.6978 Test Accuracy: 0.6974


## The model will not converge from this point no matter how deep the network is. We have tried different combinations of paramater but because of the low resolution of images, it might be hard for CNN to learn specific features no matter how many number of kernels used or how deep the network is.

## Transfer Learning
## We can use transfer learning to train VGG, ResNet, or Inception models and then train on top of it. Let's try Resnet-50 because (https://openreview.net/pdf?id=NG6MJnVl6M5) 

In [62]:
# * Preteained resnet model
tb_model = models.resnet50(weights=models.resnet.ResNet50_Weights.DEFAULT).to(device)

# * Need to redefine loaders with mean of [0.485, 0.456, 0.406] and SD of [0.229, 0.224, 0.225]
data_transforms = transforms.Compose([
    transforms.Resize((224, 224)), # * We will use 224 by 224 as the Resnet was trained on these dimensions
    transforms.ToTensor(), # * We will convert the images to tensors.
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)) # * These values were calculated from the Imagenet dataset
]
)
train_loader = torch.utils.data.DataLoader(
    datasets.ImageFolder(root='/kaggle/input/true-brands/train', transform=data_transforms),
    batch_size=128, shuffle=True, num_workers=4)

test_loader = torch.utils.data.DataLoader(
    datasets.ImageFolder(root='/kaggle/input/true-brands/test', transform=data_transforms),
    batch_size=128, shuffle=True, num_workers=4)

# * Freezing all the layers of pretrained model except the last layer because that's what we are going to train
for name, param in tb_model.named_parameters():
    if "fc" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False # * Doesn't Learn

tb_loss_function = nn.CrossEntropyLoss()
tb_optimizer = torch.optim.Adam(tb_model.parameters(), lr=0.001, weight_decay=0.0001) 

num_epochs = 10
best_accuracy = 0.0

for epoch in range(num_epochs):
    tb_model.train()
    train_accuracy = 0.0
    test_accuracy=0.0
    train_loss = 0.0

    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        tb_optimizer.zero_grad()

        x=tb_model(images)
        loss=tb_loss_function(x, labels)
        loss.backward()
        tb_optimizer.step()

        train_loss += loss.item() * images.size(0)
        _,prediction = torch.max(x, 1)

        train_accuracy += torch.sum(prediction == labels.data)
        
    train_accuracy /= train_count
    train_loss /= train_count
    
    tb_model.eval()
    
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            
            x = tb_model(images)
            _, prediction = torch.max(x, 1)
            test_accuracy += torch.sum(prediction == labels.data)
    
    test_accuracy /= test_count
    
    print(f'Epoch: {epoch} Train Loss: {train_loss:.4f} Train Accuracy: {train_accuracy:.4f} Test Accuracy: {test_accuracy:.4f}')

    
    if test_accuracy > best_accuracy:
        torch.save(tb_model.state_dict(), 'ckpt.torchmodel')
        best_accuracy = test_accuracy

Epoch: 0 Train Loss: 0.8738 Train Accuracy: 0.6537 Test Accuracy: 0.7117
Epoch: 1 Train Loss: 0.5722 Train Accuracy: 0.7137 Test Accuracy: 0.7137
Epoch: 2 Train Loss: 0.5507 Train Accuracy: 0.7224 Test Accuracy: 0.7186
Epoch: 3 Train Loss: 0.5388 Train Accuracy: 0.7260 Test Accuracy: 0.7218
Epoch: 4 Train Loss: 0.5309 Train Accuracy: 0.7301 Test Accuracy: 0.7280
Epoch: 5 Train Loss: 0.5257 Train Accuracy: 0.7332 Test Accuracy: 0.7257
Epoch: 6 Train Loss: 0.5214 Train Accuracy: 0.7338 Test Accuracy: 0.7267
Epoch: 7 Train Loss: 0.5185 Train Accuracy: 0.7375 Test Accuracy: 0.7253
Epoch: 8 Train Loss: 0.5184 Train Accuracy: 0.7369 Test Accuracy: 0.7198
Epoch: 9 Train Loss: 0.5153 Train Accuracy: 0.7397 Test Accuracy: 0.7251


## Observations:
> `The model's performance is slowly increasing over time. This was only for 10 epoch, we could see high accuracy in higher epoch values`

> `The learning becomes smaller with each epoch.`

> ` No signs of overfitting, as training and testing data are relatively moving close.`

> `Since the input images are very small in size, it is hard for model to learn feature while training. Generally, the concept is to start from big and gradually learn and compress.`


## Steps to consider:

> `Adding more data augmentation like zooming and maybe something related to smoothing the image`

> `Playing around with learning rate could result in different results.`

> `Weight Balancing can also be done considering it is not a balanced dataset`.

> `Train on some pretrained models that are trained on e-commerce dataset`




## References:

- https://viso.ai/deep-learning/pytorch-vs-tensorflow/#:~:text=It%20indicates%20a%20significantly%20higher,an%20average%20of%207.67%20seconds).
- https://stackoverflow.com/questions/61603833/how-to-recover-accidentally-deleted-cells-in-jupyter-notebook
- https://medium.com/@golnaz.hosseini/beginner-tutorial-image-classification-using-pytorch-63f30dcc071c
- https://www.kaggle.com/code/basu369victor/pytorch-tutorial-the-classification
- https://github.com/bentrevett/pytorch-image-classification
- https://www.analyticsvidhya.com/blog/2020/07/how-to-train-an-image-classification-model-in-pytorch-and-tensorflow/
- https://www.almabetter.com/bytes/articles/image-classification-using-pytorch
- https://medium.com/bitgrit-data-science-publication/building-an-image-classification-model-with-pytorch-from-scratch-f10452073212
- https://towardsdatascience.com/custom-dataset-in-pytorch-part-1-images-2df3152895
- https://www.kaggle.com/code/boascent/multi-label-image-classification-pytorch-gpu
- https://medium.com/jun94-devpblog/pytorch-1-transform-imagefolder-dataloader-7f75f0a460c0
- https://www.learndatasci.com/solutions/python-move-file/
- https://pynative.com/python-count-number-of-files-in-a-directory/
- https://towardsdatascience.com/pytorch-vision-binary-image-classification-d9a227705cf9
- https://www.youtube.com/watch?v=pDdP0TFzsoQ
- https://www.youtube.com/watch?v=ORMx45xqWkA
- https://www.youtube.com/watch?v=k1GIEkzQ8qc
- https://github.com/gaurav67890/Pytorch_Tutorials/blob/master/cnn-scratch-training.ipynb
- https://www.youtube.com/watch?v=9OHlgDjaE2I
- https://www.youtube.com/watch?v=cJpwQJp9flU
- https://datascience.stackexchange.com/questions/47328/how-to-choose-the-number-of-output-channels-in-a-convolutional-layer
- https://www.kaggle.com/code/herbison/getting-started-with-pytorch-on-cloud-tpus
- https://stackoverflow.com/questions/67472295/my-cnn-network-validation-accuracy-get-stuck-at-epoch-2-and-doesnt-change
- https://github.com/pytorch/xla#available-docker-images-and-wheels
- https://stackoverflow.com/questions/75693020/how-to-set-up-tpu-on-google-colab-for-pytorch-and-pytorch-lightning
- https://discuss.pytorch.org/t/same-accuracy-after-every-epoch/137334
- https://pytorch.org/docs/stable/nn.init.html#torch.nn.init.kaiming_uniform_
- https://datascience.stackexchange.com/questions/32651/what-is-the-use-of-torch-no-grad-in-pytorch
- https://medium.com/@kyan7472/these-are-the-5-best-pre-trained-neural-networks-23798e61a043
- https://pytorch.org/vision/stable/models.html
- https://towardsdatascience.com/the-4-convolutional-neural-network-models-that-can-classify-your-fashion-images-9fe7f3e5399d
- https://discuss.pytorch.org/t/what-does-it-mean-to-normalize-images-for-resnet/96160/3
- https://openreview.net/pdf?id=NG6MJnVl6M5
- https://stackoverflow.com/questions/48001598/why-do-we-need-to-call-zero-grad-in-pytorch
- https://medium.com/mlearning-ai/optimizers-in-deep-learning-7bf81fed78a0
- https://pytorch.org/tutorials/beginner/blitz/neural_networks_tutorial.html
- https://github.com/pytorch/vision/blob/master/torchvision/models/resnet
- https://github.com/pytorch/xla#available-docker-images-and-wheels
- https://stackoverflow.com/questions/75693020/how-to-set-up-tpu-on-google-colab-for-pytorch-and-pytorch-lightning
- https://discuss.pytorch.org/t/same-accuracy-after-every-epoch/137334
- https://pytorch.org/docs/stable/nn.init.html#torch.nn.init.kaiming_uniform_
- https://datascience.stackexchange.com/questions/32651/what-is-the-use-of-torch-no-grad-in-pytorch
- https://medium.com/@kyan7472/these-are-the-5-best-pre-trained-neural-networks-23798e61a043
- https://pytorch.org/vision/stable/models.html
- https://stackoverflow.com/questions/50204613/download-pretrained-imagenet-model-of-resnet-vgg-etc-pb-file
- https://pyimagesearch.com/2021/10/11/pytorch-transfer-learning-and-image-classification/
- https://pytorch.org/vision/main/models/generated/torchvision.models.vgg19.html
- https://towardsdatascience.com/the-4-convolutional-neural-network-models-that-can-classify-your-fashion-images-9fe7f3e5399d
- https://discuss.pytorch.org/t/what-does-it-mean-to-normalize-images-for-resnet/96160/3
- https://openreview.net/pdf?id=NG6MJnVl6M5
- https://stackoverflow.com/questions/48001598/why-do-we-need-to-call-zero-grad-in-pytorch
- https://medium.com/mlearning-ai/optimizers-in-deep-learning-7bf81fed78a0
- https://pytorch.org/docs/stable/generated/torch.Tensor.item.html
- https://datascience.stackexchange.com/questions/73944/number-of-parameters-in-resnet-50
- https://medium.com/analytics-vidhya/deep-learning-basics-weight-decay-3c68eb4344e9
- https://stackoverflow.com/questions/16910330/return-total-number-of-files-in-directory-and-subdirectories
- https://stackoverflow.com/questions/61603833/how-to-recover-accidentally-deleted-cells-in-jupyter-notebook